# 17 Void catalogs from void voxels

DeepVoid labels every voxel; a catalog needs separate voids. The T-web void class percolates (one region spans the box), so every method needs a rule for where one void ends and the next begins. None is free of choices; each choice here is an explicit parameter, and the point of this notebook is to see what each choice does, on the true labels and on the predictions alike.

| Method | Rule | Parameter |
|---|---|---|
| 1 Thresholded cores | void voxels above a score cut, or the void mask shrunk by k cells; separate pieces become voids, grown back over the void voxels | score cut or k |
| 2 Distance basins | each void voxel climbs the distance to the nearest non-void voxel to a local maximum (a void center); a basin less than Δ deeper than the pass to a bigger one is merged into it | Δ (h⁻¹ Mpc) |
| 3 Soft-distance basins | as 2, on the distance averaged over several void-score cuts, so walls the model is unsure of separate voids less | Δ, the cuts |
| 4 Maximal spheres | the distance is the largest empty sphere at each voxel; the largest spheres that barely overlap are voids, smaller ones overlapping one enough join it (the idea of VoidFinder, not its algorithm, which works on galaxies) | minimum radius, overlaps |

Method 2's merge history is a persistence diagram (section 4): each basin is born at its peak and dies at the pass where it joins a deeper one. Basins far from the diagonal are distinct voids; basins near it are dents in a bigger void. Δ is a cut on that distance.

**Data.** Bolshoi (250 h⁻¹ Mpc, 640³ cells of 0.39 h⁻¹ Mpc), never used in training, whole box. The model is the published real-space one. The truth catalog and the predicted catalogs use identical settings.

**Runtime.** Predictions (section 2) need a GPU once; they are saved to Drive and skipped afterwards. Everything else is CPU but needs **High-RAM** (the distance transform of a padded 640³ box takes ~12 GB). `COARSEN = 2` runs everything on 320³ cells for a quick look.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/deepvoid
!git clone -q "https://github.com/skumagai3/deepvoid.git" /content/deepvoid
!pip -q install --force-reinstall --no-deps /content/deepvoid
!git -C /content/deepvoid log --oneline -1

## 1 Settings

All the choices are here.

In [ ]:
import gc
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from deepvoid import Catalog
from deepvoid import voids as V
from deepvoid.labels import VOID
from deepvoid.resample import block_mean

RUNS = Path('/content/drive/MyDrive/deepvoid_runs')
MODEL = RUNS.parent / 'deepvoid_model_release'       # notebook 11
OUT = RUNS / 'voids'
OUT.mkdir(exist_ok=True)
catalog = Catalog()

BOX = 'bolshoi'
SPACINGS = [3, 10]
COARSEN = 1                 # 2: 320^3 cells, for a quick look
MARGIN = 96                 # padding (cells) for the periodic distance transform
R_CUT = 3.0                 # h^-1 Mpc: voids listed and matched have R_eff above this

ERODE = [1, 2, 3]                       # method 1, mask shrunk by k cells
SCORE_CUTS = [0.8, 0.95]                # method 1, void score cuts (predictions only)
PERSISTENCE = [0.5, 1.0, 2.0]           # methods 2 and 3, Δ in h^-1 Mpc
MIN_RADIUS = 1.0                        # methods 2 and 3: drop voids whose largest empty sphere is smaller
SOFT_CUTS = (0.2, 0.5, 0.8)             # method 3, void score cuts averaged over
SPHERES = [dict(r_min=5.0), dict(r_min=10.0)]   # method 4; overlaps 10% / 50% as VoidFinder

## 2 Truth and predictions

The predictions are saved once (void score and class per voxel) and read back afterwards.

In [ ]:
def coarse_mask(void):
    return block_mean(void.astype(np.float32), COARSEN) >= 0.5 if COARSEN > 1 else void


truth_vol = catalog.load(f'{BOX}/mask')
box = truth_vol.box_size
truth_void = coarse_mask(truth_vol.data == VOID)
n = truth_void.shape[0]
cell = box / n
del truth_vol
print(f'{BOX}: {n}^3 cells of {cell:.3f} h^-1 Mpc; void share {truth_void.mean():.3f}')

pred = {}
for s in SPACINGS:
    path = OUT / f'{BOX}_L{s}_scores.npz'
    if not path.exists():
        from deepvoid.pretrained import Pretrained
        model = globals().get('model') or Pretrained.from_folder(MODEL)
        vol = catalog.load(f'{BOX}/halos/L{s}')
        t0 = time.time()
        probs = model.predict(vol.data, vol.box_size, float(s), out_dtype=np.float16)
        np.savez(path, score=probs[..., VOID], classes=probs.argmax(-1).astype(np.uint8))
        print(f'λ = {s}: predicted in {time.time() - t0:.0f} s, saved {path.name}')
        del probs, vol
        gc.collect()
    data = np.load(path)
    score = data['score'].astype(np.float32)
    void = data['classes'] == VOID
    if COARSEN > 1:
        score = block_mean(score, COARSEN)
        void = coarse_mask(void)
    pred[s] = {'score': score, 'void': void}
    print(f'λ = {s}: void share {void.mean():.3f}, agreement with truth {np.mean(void == truth_void):.3f}')

## 3 Does the void class percolate?

Number of separate void regions (periodic) and the share of void voxels in the largest. If the largest holds nearly everything, connected components alone give one giant void.

In [ ]:
perc = {'truth': V.percolation(truth_void)}
for s in SPACINGS:
    perc[f'λ = {s}'] = V.percolation(pred[s]['void'])
print(pd.DataFrame(perc).T.to_string())

## 4 Distances and persistence diagrams

Distance from every void voxel to the nearest non-void voxel (h⁻¹ Mpc), then every basin of that distance with its birth (peak) and death (the pass where it joins a deeper basin). The vertical distance below the diagonal is how much deeper the basin is than the pass: its persistence. Δ in methods 2 and 3 keeps basins with persistence above Δ.

In [ ]:
t0 = time.time()
dist = {'truth': V.distance_to_structure(truth_void, cell=cell, margin=MARGIN)}
for s in SPACINGS:
    dist[s] = V.distance_to_structure(pred[s]['void'], cell=cell, margin=MARGIN)
print(f'distances in {time.time() - t0:.0f} s; largest:', {k: round(float(d.max()), 1) for k, d in dist.items()})

diagrams = {'truth': V.basin_persistence(truth_void, dist['truth'])}
for s in SPACINGS:
    diagrams[s] = V.basin_persistence(pred[s]['void'], dist[s])

fig, axes = plt.subplots(1, len(diagrams) + 1, figsize=(4.2 * (len(diagrams) + 1), 4))
for ax, (k, h) in zip(axes, diagrams.items()):
    ax.hexbin(h[:, 0], h[:, 1], gridsize=60, bins='log', mincnt=1, cmap='viridis')
    lim = max(h[:, 0].max(), 1)
    ax.plot([0, lim], [0, lim], 'k-', lw=0.8)
    for p in PERSISTENCE:
        ax.plot([p, lim], [0, lim - p], 'r:', lw=0.8)
    ax.set_title('truth' if k == 'truth' else f'prediction, λ = {k}')
    ax.set_xlabel('birth: basin peak (h⁻¹ Mpc)'); ax.set_ylabel('death: pass (h⁻¹ Mpc)')
ax = axes[-1]
bins = np.logspace(-2, 1.3, 60)
for k, h in diagrams.items():
    ax.hist(h[:, 0] - h[:, 1], bins=bins, histtype='step', label='truth' if k == 'truth' else f'λ = {k}')
for p in PERSISTENCE:
    ax.axvline(p, color='r', ls=':', lw=0.8)
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('persistence (h⁻¹ Mpc)'); ax.legend()
ax.set_title('persistence of every merge')
fig.tight_layout()
fig.savefig(OUT / f'{BOX}_persistence.png', dpi=150, bbox_inches='tight')
plt.show()

## 5 Catalogs from every method

Each method runs on the truth and on each prediction with the same settings. For each catalog: the number of voids with R_eff > `R_CUT`, their median R_eff, and, for the predictions, how they match the truth catalog made with the same settings:

- **completeness**: share of true voids (R_eff > `R_CUT`) whose best-overlapping predicted void shares more than half their union (Jaccard > 0.5);
- **purity**: the same from the predicted side;
- **offset**: median distance between matched centers, h⁻¹ Mpc.

For method 4 (spheres), voids match when the predicted void's largest sphere is centered within half the true one's radius.

In [ ]:
def periodic_sep(a, b):
    d = a - b
    d -= box * np.round(d / box)
    return np.sqrt((d ** 2).sum(-1))


def compare(lab_t, lab_p):
    cat_t, cat_p = V.void_catalog(lab_t, cell), V.void_catalog(lab_p, cell)
    big_t, big_p = cat_t['r_eff'] > R_CUT, cat_p['r_eff'] > R_CUT
    tp, pt = V.match_voids(lab_t, lab_p), V.match_voids(lab_p, lab_t)
    ok_t = big_t & (tp['jaccard'] > 0.5)
    ok_p = big_p & (pt['jaccard'] > 0.5)
    off = periodic_sep(cat_t['center'][ok_t], cat_p['center'][tp['best'][ok_t] - 1])
    return cat_p, {'voids': int(big_p.sum()), 'median R_eff': float(np.median(cat_p['r_eff'][big_p])) if big_p.any() else np.nan,
                   'completeness': float(ok_t.sum() / max(big_t.sum(), 1)),
                   'purity': float(ok_p.sum() / max(big_p.sum(), 1)),
                   'offset': float(np.median(off)) if len(off) else np.nan}


def summary(lab):
    cat = V.void_catalog(lab, cell)
    big = cat['r_eff'] > R_CUT
    return cat, {'voids': int(big.sum()), 'median R_eff': float(np.median(cat['r_eff'][big])) if big.any() else np.nan}


def methods():
    """(name, function of (void, score, distance)); score is None for the truth."""
    for k in ERODE:
        yield f'1 erode {k}', lambda v, sc, d, k=k: V.threshold_voids(v, erode=k)
    for c in SCORE_CUTS:
        yield f'1 score ≥ {c}', lambda v, sc, d, c=c: None if sc is None else V.threshold_voids(v, score=sc, threshold=c)
    for p in PERSISTENCE:
        yield f'2 Δ = {p}', lambda v, sc, d, p=p: V.distance_voids(v, distance=d, persistence=p, min_radius=MIN_RADIUS)


rows, radii = {}, {}
for name, run in methods():
    t0 = time.time()
    lab_t = run(truth_void, None, dist['truth'])
    if lab_t is not None:
        cat_t, rows[(name, 'truth')] = summary(lab_t)
        radii[(name, 'truth')] = cat_t['r_eff']
    for s in SPACINGS:
        lab_p = run(pred[s]['void'], pred[s]['score'], dist[s])
        if lab_t is None:
            cat_p, rows[(name, f'λ = {s}')] = summary(lab_p)
        else:
            cat_p, rows[(name, f'λ = {s}')] = compare(lab_t, lab_p)
        radii[(name, f'λ = {s}')] = cat_p['r_eff']
        del lab_p
    del lab_t
    gc.collect()
    print(f'{name}: {time.time() - t0:.0f} s', flush=True)

Method 3: the soft distance exists only for predictions (the truth has no scores), so its catalogs are compared with the truth's method-2 catalog at the same Δ. Its distances are averages over the cuts, so the same Δ is not quite the same depth.

In [ ]:
soft = {}
for s in SPACINGS:
    t0 = time.time()
    soft[s] = V.soft_distance(pred[s]['score'], thresholds=SOFT_CUTS, cell=cell, margin=MARGIN)
    print(f'λ = {s}: soft distance in {time.time() - t0:.0f} s', flush=True)
for p in PERSISTENCE:
    lab_t = V.distance_voids(truth_void, distance=dist['truth'], persistence=p, min_radius=MIN_RADIUS)
    for s in SPACINGS:
        lab_p = V.distance_voids(pred[s]['void'], distance=soft[s], persistence=p, min_radius=MIN_RADIUS)
        cat_p, rows[(f'3 Δ = {p}', f'λ = {s}')] = compare(lab_t, lab_p)
        radii[(f'3 Δ = {p}', f'λ = {s}')] = cat_p['r_eff']
        del lab_p
    del lab_t
    gc.collect()
del soft

In [ ]:
def sphere_match(sp_t, sp_p):
    """True voids whose largest sphere has a predicted largest sphere within half its radius."""
    def largest(sp):
        ids = np.unique(sp['void'])
        out_c, out_r = [], []
        for v in ids:
            sel = sp['void'] == v
            k = np.argmax(sp['radius'] * sel)
            out_c.append(sp['center'][k]); out_r.append(sp['radius'][k])
        return np.array(out_c).reshape(-1, 3), np.array(out_r)
    ct, rt = largest(sp_t)
    cp, rp = largest(sp_p)
    if not len(ct) or not len(cp):
        return 0.0, 0.0, np.nan
    tree_p, tree_t = cKDTree(cp % box, boxsize=box), cKDTree(ct % box, boxsize=box)
    dt, _ = tree_p.query(ct % box)
    dp, _ = tree_t.query(cp % box)
    ok_t = dt < rt / 2
    return float(ok_t.mean()), float(np.mean(dp < rp / 2)), float(np.median(dt[ok_t])) if ok_t.any() else np.nan


for kw in SPHERES:
    name = f"4 spheres r_min = {kw['r_min']}"
    t0 = time.time()
    sp_t = V.sphere_voids(dist['truth'], truth_void, cell, **kw)
    vol_t = V.sphere_union_volume(sp_t)
    r_t = (3 * vol_t / (4 * np.pi)) ** (1 / 3)
    rows[(name, 'truth')] = {'voids': len(r_t), 'median R_eff': float(np.median(r_t)) if len(r_t) else np.nan}
    radii[(name, 'truth')] = r_t
    for s in SPACINGS:
        sp_p = V.sphere_voids(dist[s], pred[s]['void'], cell, **kw)
        r_p = (3 * V.sphere_union_volume(sp_p) / (4 * np.pi)) ** (1 / 3)
        comp, pur, off = sphere_match(sp_t, sp_p)
        rows[(name, f'λ = {s}')] = {'voids': len(r_p), 'median R_eff': float(np.median(r_p)) if len(r_p) else np.nan,
                                    'completeness': comp, 'purity': pur, 'offset': off}
        radii[(name, f'λ = {s}')] = r_p
    print(f'{name}: {time.time() - t0:.0f} s', flush=True)

## 6 Results

In [ ]:
table = pd.DataFrame(rows).T
table.index.names = ['method', 'catalog']
pd.set_option('display.width', 160)
print(table.round(2).to_string())
(OUT / f'{BOX}_results.json').write_text(json.dumps(
    {f'{m}|{c}': {k: (None if pd.isna(v) else v) for k, v in r.items()} for (m, c), r in rows.items()}, indent=1))
np.savez(OUT / f'{BOX}_radii.npz', **{f'{m}|{c}': r for (m, c), r in radii.items()})

In [ ]:
names = list(dict.fromkeys(m for m, _ in radii))
ncol = 4
nrow = int(np.ceil(len(names) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.4 * nrow), sharex=True, sharey=True)
bins = np.linspace(R_CUT, 40, 80)
styles = {'truth': ('k', '-')}
styles.update({f'λ = {s}': (c, '--') for s, c in zip(SPACINGS, ['#dd6b20', '#2b6cb0', '#38a169'])})
for ax, name in zip(axes.ravel(), names):
    for (m, c), r in radii.items():
        if m == name:
            col, ls = styles[c]
            ax.step(bins, V.size_function(r, box, bins), where='post', color=col, ls=ls, label=c)
    if name.startswith('3 ') and ('2 ' + name[2:], 'truth') in radii:   # method 3: truth from method 2
        ax.step(bins, V.size_function(radii[('2 ' + name[2:], 'truth')], box, bins), where='post',
                color='k', ls='-', label='truth (method 2)')
    ax.set_yscale('log'); ax.set_title(name, fontsize=10)
for ax in axes.ravel()[len(names):]:
    ax.axis('off')
for ax in axes[-1]:
    ax.set_xlabel(r'$R_{\rm eff}$ ($h^{-1}$ Mpc)')
for ax in axes[:, 0]:
    ax.set_ylabel(r'$n(>R)$ ($h^3$ Mpc$^{-3}$)')
axes.ravel()[0].legend(fontsize=8)
fig.suptitle(f'Void size functions, {BOX}: truth against predictions, same settings')
fig.tight_layout()
fig.savefig(OUT / f'{BOX}_size_functions.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 What the voids look like

One slice through method 2 at the middle Δ: truth and the λ = 10 prediction. Each void gets a random color; structure is gray. Colors are not matched between panels.

In [ ]:
P = PERSISTENCE[len(PERSISTENCE) // 2]
SLICE = n // 2
s = SPACINGS[-1]
lab_t = V.distance_voids(truth_void, distance=dist['truth'], persistence=P, min_radius=MIN_RADIUS)
lab_p = V.distance_voids(pred[s]['void'], distance=dist[s], persistence=P, min_radius=MIN_RADIUS)
rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, lab, title in ((axes[0], lab_t, 'truth'), (axes[1], lab_p, f'prediction, λ = {s}')):
    sl = lab[SLICE]
    colors = rng.random((sl.max() + 1, 3)) * 0.8 + 0.2
    colors[0] = 0.25
    ax.imshow(colors[sl].transpose(1, 0, 2), origin='lower', extent=[0, box, 0, box])
    ax.set_title(f'{title}: method 2, Δ = {P} h⁻¹ Mpc')
    ax.set_xlabel('y (h⁻¹ Mpc)'); ax.set_ylabel('z (h⁻¹ Mpc)')
fig.tight_layout()
fig.savefig(OUT / f'{BOX}_slice_delta{P}.png', dpi=150, bbox_inches='tight')
plt.show()
del lab_t, lab_p
print('saved in', OUT)

## Shut down

Deletes this runtime once every cell above has finished, so it stops using compute units. The results are already on Drive; anything kept only under `/content` goes with the runtime. With **Run all**, Colab stops at the first cell that fails, so after an error the runtime stays up and you can look at it. Set `SHUT_DOWN = False` to keep the runtime.

In [ ]:
SHUT_DOWN = True   # False keeps the runtime after the run
if SHUT_DOWN:
    import time
    from google.colab import drive, runtime
    drive.flush_and_unmount()   # finish writing to Drive first
    time.sleep(5)
    runtime.unassign()          # disconnects and deletes this runtime